In [4]:
with open('PCmaster_anno_0_23_12_25.py','r',encoding='utf-8') as f:
    exec(f.read())

2024-04-23 11:51:54
available_memory: 21.64618682861328 GB
used_memory: 10.209339141845703 GB


In [5]:
def print_memory_use():
    gc.collect()
    available_memory = psutil.virtual_memory().available
    available_memory_gb = available_memory / (1024 * 1024 * 1024)
    print(f"available_memory: {available_memory_gb} GB")
    used_memory = psutil.virtual_memory().used
    used_memory_gb = used_memory / (1024 * 1024 * 1024)
    print(f"used_memory: {used_memory_gb} GB")
    process = psutil.Process()
    memory_usage_bytes = process.memory_info().rss
    memory_usage_gb = memory_usage_bytes / 1024.0 / 1024.0 / 1024.0
    print(f"used_memory_in_this: {memory_usage_gb} GB")

In [6]:
print_memory_use()

available_memory: 21.646900177001953 GB
used_memory: 10.208625793457031 GB
used_memory_in_this: 0.5067024230957031 GB


In [7]:
# 5
# grep a batch of k-mer for test
    
# def get_k_mer(input,output,num,model='intersection'):
#     '''
#     input is a list,
#     if model is intersection, function will do intersection to k-mer in input,
#     if model is union, function will do union to k-mer in input,
#     return a k-mer-index dict,
#     '''
#     if len(input)>1 and model=='intersection':
#         

In [8]:
def sequence_count_varieties(file_path):  
    sequence_count_dict = {}
    with open(file_path, 'r', encoding='utf-8') as file:  
        while True:  
            try:  
                count = int(next(file).strip().replace('>',''))
                sequence = next(file).strip()  
                sequence_count_dict[sequence] = count
            except StopIteration:  
                break  
    return sequence_count_dict

In [9]:
def get_top_k_mers(sequence_count_dict_paths, max_output_num_of_k_mer, model='intersection'):
    '''
    Select the sequence that appears in all dictionaries and has the highest total count,
    input is a list,
    if model is intersection, function will do intersection to k-mer in input,
    if model is union, function will do union to k-mer in input,
    return a k-mer-index dict,
    '''
    print_memory_use()
    start_time = time.time()
    dict_paths_len = len(sequence_count_dict_paths)
    total_counts = {}
    total_appear = {}
    for sequence_count_dict_path in sequence_count_dict_paths:
        if isinstance(sequence_count_dict_path,str) and sequence_count_dict_path.endswith('mer_counts_dumps.fa'):
            sequence_count_dict = sequence_count_varieties(file_path=sequence_count_dict_path)
        else:
            sequence_count_dict = sequence_count_dict_path
        for sequence, count in sequence_count_dict.items():
            total_counts[sequence] = total_counts.get(sequence, 0) + count
            if model=='intersection':
                total_appear[sequence] = total_appear.get(sequence, 0) + 1
            # gc.collect()
            # available_memory = psutil.virtual_memory().available
            # available_memory_gb = available_memory / (1024 * 1024 * 1024)
            # used_memory = psutil.virtual_memory().used
            # used_memory_gb = used_memory / (1024 * 1024 * 1024)
            # if used_memory_gb > 0.95*available_memory_gb:
            #     print('used_memory_gb > 0.95*available_memory_gb, break!')
            #     break
        del sequence_count_dict
        gc.collect()
    total_counts = {k: v for k, v in sorted(total_counts.items(), key=lambda item: item[1], reverse=True)}
    sequences_in_all_dicts = []
    get_count = 0
    for sequence in total_counts:
        if model=='intersection' and total_appear[sequence] == dict_paths_len:
            sequences_in_all_dicts.append(sequence)
            get_count = get_count+1
        elif model=='union':
            sequences_in_all_dicts.append(sequence)
            get_count = get_count+1
        if get_count == max_output_num_of_k_mer:
            break
    end_time = time.time()
    print(f'time past: {(end_time-start_time)/60} min')
    print_memory_use()
    del total_counts
    gc.collect()
    return sequences_in_all_dicts

In [10]:
sequence_count_dicts = [
    {'ATCG': 10, 'CGTA': 5, 'GCTA': 8},
    {'ATCG': 1, 'CGTA': 7, 'GCTA': 6},
    {'ATCG': 1, 'CGTA': 3,}
]

top_sequences = get_top_k_mers(
    sequence_count_dict_paths=sequence_count_dicts, 
    max_output_num_of_k_mer=3, model='intersection')
print(top_sequences)

available_memory: 21.650428771972656 GB
used_memory: 10.205680847167969 GB
used_memory_in_this: 0.5067100524902344 GB
time past: 0.005776747067769369 min
available_memory: 21.64293670654297 GB
used_memory: 10.212589263916016 GB
used_memory_in_this: 0.5067100524902344 GB
['CGTA', 'ATCG']


In [11]:
tmp_paths = [
    'D:/data-from-e/Root-1_S9_L001_R2_001_extracted_use_umi_to_make_umi_unique_12-mer_counts_dumps.fa',
    'D:/data-from-e/SRR14299279_S1_L001_R2_001_extracted_use_umi_to_make_umi_unique_12-mer_counts_dumps.fa',
]
top_sequences = get_top_k_mers(
    sequence_count_dict_paths=tmp_paths, 
    max_output_num_of_k_mer=50000, model='intersection')
print(len(top_sequences))

available_memory: 21.64168930053711 GB
used_memory: 10.213836669921875 GB
used_memory_in_this: 0.5067100524902344 GB
time past: 0.47983632882436117 min
available_memory: 20.482418060302734 GB
used_memory: 11.37310791015625 GB
used_memory_in_this: 2.152606964111328 GB
50000


In [12]:
# check memory many times will cost much time!!!

In [13]:
k_mer_index_dict = {}
index = 0
for i in top_sequences:
    k_mer_index_dict[i] = index
    index = index+1
del top_sequences
gc.collect()
print_memory_use()

k_mer_lens = [12]

0

available_memory: 21.538116455078125 GB
used_memory: 10.31740951538086 GB
used_memory_in_this: 1.1140289306640625 GB


In [14]:
print(k_mer_index_dict)

{'AAAAAAAAAAAA': 0, 'ATCAACGCAGAG': 1, 'AACGCAGAGTAC': 2, 'CAACGCAGAGTA': 3, 'ACGCAGAGTACA': 4, 'ACTCTGCGTTGA': 5, 'ATGTACTCTGCG': 6, 'CATGTACTCTGC': 7, 'CAGAGTACATGG': 8, 'TATCAACGCAGA': 9, 'CTGCGTTGATAC': 10, 'GGTATCAACGCA': 11, 'AGAGTACATGGG': 12, 'GCGTTGATACCA': 13, 'CGTTGATACCAC': 14, 'CCCCCCCCCCCC': 15, 'AGTGGTATCAAC': 16, 'CAGTGGTATCAA': 17, 'GCAGTGGTATCA': 18, 'AGCAGTGGTATC': 19, 'AAGCAGTGGTAT': 20, 'CCCCATGTACTC': 21, 'CCGCCGCCGCCG': 22, 'GCCGCCGCCGCC': 23, 'CGCCGCCGCCGC': 24, 'AGTACATGGGGG': 25, 'GAGTACATGGGC': 26, 'GAGTACATGGGA': 27, 'AGTACATGGGGA': 28, 'CAAAAAAAAAAA': 29, 'TAAAAAAAAAAA': 30, 'AGTACATGGGGC': 31, 'GAGAGAGAGAGA': 32, 'ACCCCATGTACT': 33, 'ACCCATGTACTC': 34, 'AGAGAGAGAGAG': 35, 'GAAAAAAAAAAA': 36, 'AAGAAGAAGAAG': 37, 'GAAGAAGAAGAA': 38, 'AGAAGAAGAAGA': 39, 'CAGCAGCAGCAG': 40, 'AGTACATGGGCG': 41, 'AGTACATGGGAG': 42, 'ACGACGACGACG': 43, 'ATCGATCGATCG': 44, 'AGTACATGGGCA': 45, 'CCGGCGGCGGCG': 46, 'AGAAAAAAAAAA': 47, 'TTAAAAAAAAAA': 48, 'GCAGCAGCAGCA': 49, 'ATAAAAAA

In [15]:
# with open('k_mer_index_dict_len_12_24_2_22.pkl','wb') as file:
#     pickle.dump(k_mer_index_dict,file)